In [ ]:
import logging
import os
import json
from pathlib import Path
import configparser

from core.db.crud import DatabaseManager
from core.dellattachments import DellAttachments
from core.dispatcher import Dispatcher
from services.classification.classification.classification import Classifier
from config import Config as cfg
logger = logging.getLogger(__name__)

async def process_document(payload: dict):
    doc_dict = {
        'requestid': payload.get('uuid'),
        'daoriginal_fileid': payload.get('daFileId'),
        'dafileid': payload.get('daFileId'),
        'filename': payload.get('name'),
        'description': payload.get('description'),
        'dtpm_phase': payload.get('phase'),
        'ip_type': payload.get('ipTypes'),
        'document_type': payload.get('source'),
        'offer': payload.get('offer'),
        'status': payload.get('status'),
        'ipid': payload.get('ipId'),
        'initialgrade': payload.get('initialGrade'),
        'priority': payload.get('priority'),
        'created_by': 'system'
    }
    
    db = DatabaseManager()
    db.insert_document(**doc_dict)
    logger.info(f"Document inserted into DB")
    
    file_dict = {
        'id': payload.get('daFileId'),
        'name': payload.get('name'),
        'filepath': os.path.join(
            cfg.INDIR,
            str(payload.get('uuid')),
            payload.get('name')
        )
    }
    
    da = DellAttachments()
    file_dict = await da.download(filedict=file_dict)
    logger.info(f"File downloaded from Dell Attachments")
    
    filepath = file_dict['filepath']
    fileid = file_dict['id']
    
    dispatcher = Dispatcher(filepath, fileid,analyze_images=cfg.IMAGE_ANALYZE_SWITCH,debug=cfg.DEBUG)
    text_content = dispatcher.getExtractor().extract_content()
    
    classifier = Classifier(fileid, text_content)
    result = classifier.classify()
    
    db.update_document(
        where_clause={
            'requestid': doc_dict['requestid'],
            'daoriginal_fileid': doc_dict['daoriginal_fileid']
        },
        update_values={
            'title': result.get('title'),
            'description': result.get('description')
        }
    )
    logger.info(f"Document updated with title and description")
    
    return {
        'daoriginal_fileid': fileid,
        'title': result.get('title'),
        'description': result.get('description')
    }

def load_payload(payload_file: str = 'dummy_payload.json'):
    with open(payload_file, 'r') as f:
        data = json.load(f)
    return data['payload']

async def main():
    payload = load_payload()
    logger.info(f"Processing document: {payload.get('name')}")
    result = await process_document(payload)
    logger.info(f"Process completed successfully")
    logger.info(f"Result: {result}")
    return result

if __name__ == "__main__":
    import asyncio
    try:
        loop = asyncio.get_running_loop()
        import nest_asyncio
        nest_asyncio.apply()
        asyncio.run(main())
    except RuntimeError:
        asyncio.run(main())